In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler

In [6]:
pokemon_df = pd.read_csv('C:/Users/Carlos/Documents/DarkDex/darkdex/archives/Pokemon.csv')

In [7]:
pokemon_df = pokemon_df.rename(
    columns={
        "#": "id",
        "Name": "name",
        "Attack": "attack",
        "Defense": "defense",
        "Sp. Atk": "sp_attack",
        "Sp. Def": "sp_defense",
        "HP": "hp",
        "Speed": "speed",
    }
)

In [8]:
pokemon_df["offensive_power"] = pokemon_df["attack"] + pokemon_df["sp_attack"]
pokemon_df["defensive_power"] = pokemon_df["defense"] + pokemon_df["sp_defense"]
pokemon_df["bulk"] = pokemon_df["hp"] + pokemon_df["defensive_power"]
pokemon_df["speed_score"] = pokemon_df["speed"]

In [9]:
counts_df = pokemon_df.groupby('id').size().reset_index(name='Total_Count')

In [10]:
pokemon_df = pokemon_df[
    ~pokemon_df["name"].str.contains(
        "Primal|Gigantamax|Ultra|Gmax|Alolan|Galarian|Hisuian|Mega|Shadow|Origin|Totem|Therian|Eternamax|Rotom|Forme",
        case=False,
        na=False,
    )
]

In [11]:
def plot_distribution(df, identificacao, x, y=None, z=None, color=None, jitter_1d=0.02):

    df = df.copy()

    if z is not None and y is not None:
        fig = px.scatter_3d(
            df,
            x=x,
            y=y,
            z=z,
            color=color,
            opacity=0.8,
            hover_name=identificacao,
            title="Visualização 3D Interativa"
        )

    elif y is not None:
        fig = px.scatter(
            df,
            x=x,
            y=y,
            color=color,
            opacity=0.8,
            hover_name=identificacao,
            title="Visualização 2D Interativa"
        )

    else:
        df["_y_dummy"] = np.random.uniform(
            low=-jitter_1d,
            high=jitter_1d,
            size=len(df)
        )

        fig = px.scatter(
            df,
            x=x,
            y="_y_dummy",
            color=color,
            opacity=0.8,
            hover_name=identificacao,
            title="Visualização 1D Interativa"
        )

        fig.update_yaxes(visible=False)

    fig.update_traces(marker=dict(size=7))
    fig.update_layout(
        width=900,
        height=600,
        legend_title=color
    )

    fig.show()

In [12]:
plot_distribution(pokemon_df, 'name', 'offensive_power')

In [13]:
def definir_clusters_cotovelo(df_kmeans, columns):
    scaler = StandardScaler()

    df_scaled = scaler.fit_transform(df_kmeans[columns])

    inertia = []
    for n_clusters in range(1, 11):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans.fit(df_scaled)
        inertia.append(kmeans.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, 11), inertia, marker='o')
    plt.xlabel('Número de Clusters')
    plt.ylabel('Inércia')
    plt.title('Método do Cotovelo com RobustScaler')
    plt.show()

    return df_kmeans, df_scaled